# OCR Bilans Fiscaux Algériens — V4 — H100 + Qwen3.6-VL-27B

> **Objectif** : extraire les bilans fiscaux PDF → JSON complet + Excel récapitulatif + fichiers FORFAIT `.xlsx` par client/année.

## Nouveautés V4
| # | Changement | Détail |
|---|---|---|
| 1 | **TCR DEBIT/CREDIT** | 4 colonnes : `debit_n`, `credit_n`, `debit_n1`, `credit_n1` |
| 2 | **Bloc DECL enrichi** | ~30 champs : NIF, raison sociale, activité, CAC, CA, résultat comptable/fiscal, période… |
| 3 | **Contrôles renforcés** | Sommes par section, brut−amort=net, cohérence TCR, DECL↔TCR↔Passif |
| 4 | **Générateur FORFAIT** | Cellule 14 : copie template `.xlsx` + remplissage Saisie actif/passif/TCR + formules aval intactes |

## Livrables
| # | Livrable | Format | Contenu |
|---|---|---|---|
| 1 | Base de données bilans | `.json` par PDF | ACTIF, PASSIF, TCR (D/C), DECL, contrôles, anomalies |
| 2 | Vue analyste | `.xlsx` | 2 lignes/bilan (N, N-1), colonnes META + ACTIF + PASSIF + TCR + DECL |
| 3 | FORFAIT par client | `FORFAIT Cas 1_{client}_{année}.xlsx` | Saisie actif/passif/TCR remplies, formules `Etats fin reclassés` et `Décision` actives |

## Règle d'or
**Ne jamais inventer de valeur** : toute donnée absente/incertaine → `null` + anomalie signalée.

## Journal des versions
| Version | Date | Changement |
|---|---|---|
| V1 | 2026-08-14 | Première version |
| V2 | 2026-08-14 | 2 lignes/an, deskew, contrôles cohérence |
| V3 | 2026-08-14 | Two-pass (classification + extraction) |
| V4 | 2026-08-14 | TCR DEBIT/CREDIT, DECL enrichi, contrôles renforcés, générateur FORFAIT `.xlsx` |

## Cellule 1 — Dépendances
**Objectif** : installer le runtime.
**Contrainte** : `transformers >= 4.57` requis (architecture Qwen3-VL + intégration FP8 `FineGrainedFP8Config`).
**V4** : `openpyxl` pour le générateur FORFAIT `.xlsx`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DÉPENDANCES | Projet : BILANS_V4 | 2026-08-14
# Rôle : runtime OCR + génération FORFAIT (.xlsx)
# ════════════════════════════════════════════════════════════
%pip install -q -U "transformers>=4.57.0" accelerate pymupdf pillow openpyxl psutil pandas
print('✅ OK')

## Cellule 2 — Imports

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS | BILANS_V4 | 2026-08-14
# ════════════════════════════════════════════════════════════
import time, json, re, gc, difflib, csv, shutil
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

## Cellule 3 — Config
**Contient `FORFAIT_COORDS`** : coordonnées des cellules dans le template FORFAIT.
- Indices **1-based** (format openpyxl : row=1, column=1 = A1)
**Exécutez la Cellule 16** pour vérifier/ajuster ces coordonnées avant la première génération.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG | BILANS_V4 | 2026-08-14
# Entrées : chemins ModelHub / data / template FORFAIT
# Sorties : pdfs, EXCEL_PATH, FORFAIT_OUTPUT_DIR
# ════════════════════════════════════════════════════════════
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'  # ← AJUSTER

DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 3500
IMAGE_MAX_SIZE = 2024
MIN_PIXELS     = 4 * 32 * 32
MAX_PIXELS     = 2000 * 32 * 32
PDF_ZOOM       = 3.0
BLANK_THRESHOLD= 0.95
GPU_BATCH_SIZE = 8
ANNEE_ATTENDUE = None  # ex '2025' → anomalie si dossier d'une autre année

INPUT_DIR  = Path('/mnt/data/bilans_in')
OUTPUT_DIR = Path('/mnt/data/bilans_out')
JSON_DIR   = OUTPUT_DIR / 'json_bilans'
LOG_PATH   = OUTPUT_DIR / 'pipeline_bilans.log'
EXCEL_PATH = OUTPUT_DIR / f'bilans_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

# ─── FORFAIT .xlsx (V4) ─────────────────────────────────────
TEMPLATE_FORFAIT   = Path('/mnt/data/forfait/FORFAIT Cas 1 TARZAALI.xlsx')  # ← AJUSTER
FORFAIT_OUTPUT_DIR = OUTPUT_DIR / 'forfaits'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
FORFAIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))

# ════════════════════════════════════════════════════════════
# FORFAIT_COORDS — indices 1-BASED (openpyxl)
# À VÉRIFIER avec la Cellule 16 avant la première exécution
# ════════════════════════════════════════════════════════════
FORFAIT_COORDS = {
    'actif': {
        'sheet': 'Saisie actif',
        # Colonnes : N (brut, amort, net) | N-1 (brut, amort, net)
        'cols_n':  {'brut': 5, 'amort': 6, 'net': 7},
        'cols_n1': {'brut': 8, 'amort': 9, 'net': 10},
        # En-tête
        'nature_bilan':   (2, 5),   # cellule 'Fiscal'
        'date_arrete_n':  (3, 5),
        'date_arrete_n1': (3, 8),
        'certifie_cac':   (4, 10),
        # Recherche par libellé (colonne des libellés = colonne 1)
        'label_col': 1,
    },
    'passif': {
        'sheet': 'Saisie passif',
        'col_n':  3,
        'col_n1': 4,
        'label_col': 2,
    },
    'tcr': {
        'sheet': 'Saisie TCR',
        # Colonnes N : DEBIT(KDZD), CREDIT(KDZD) | N-1 : DEBIT, CREDIT
        'cols_n':  {'debit': 5, 'credit': 6},
        'cols_n1': {'debit': 7, 'credit': 8},
        'label_col': 1,
    },
}

print(f'Device : {DEVICE} | Dossiers : {len(pdfs)} | Excel : {EXCEL_PATH}')
print(f'Template FORFAIT : {TEMPLATE_FORFAIT} | Sortie : {FORFAIT_OUTPUT_DIR}')

## Cellule 4 — Chargement modèle
**Décision technique** : le checkpoint FP8 block-wise exige le kernel `kernels-community/finegrained-fp8`, indisponible en offline → `FineGrainedFP8Config(dequantize=True)` convertit les poids en bf16 au load. Coût : ~54 GB VRAM (H100 80GB).
**Réglages** : `padding_side='left'` (génération batch) ; `enable_thinking=False` (extraction déterministe).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — CHARGEMENT MODÈLE | BILANS_V4 | 2026-08-14
# FP8 → bf16 (dequantize) | padding gauche | thinking off
# ════════════════════════════════════════════════════════════
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True,
                                          min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'

try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto',
    trust_remote_code=True, low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s | VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## Cellule 5 — Utilitaires PDF & inférence
- `deskew` : méthode du profil de projection (±12°, pas 2°, puis raffinement ±1° pas 0.5°)
- `ask_batch` : N images / 1 appel GPU, décodage greedy (reproductible)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — UTILITAIRES | BILANS_V4 | 2026-08-14
# deskew (profil de projection) + inférence batch reproductible
# ════════════════════════════════════════════════════════════
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img

def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        pages.append({'index': i, 'image': resize(deskew(Image.frombytes('RGB', [pix.width, pix.height], pix.samples)))})
    doc.close()
    return pages

def parse_json(text: str) -> dict:
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}

def apply_template(messages) -> str:
    try:
        return processor.apply_chat_template(messages, tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True,
                            clean_up_tokenization_spaces=False)

def ask_single(prompt, image) -> dict:
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]),
            'tokens_in': int(inputs['input_ids'].shape[1]),
            'tokens_out': int(out[0].shape[0] - inputs['input_ids'].shape[1]),
            'elapsed': round(time.time()-t0, 2)}

def ask_batch(prompt, images) -> list:
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images,
                       return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len),
             'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len,
             'tokens_out': int(out[i].shape[0] - in_len), 'elapsed': round(el/len(images), 2)}
            for i in range(len(images))]

print('✅ Utilitaires OK (deskew inclus)')

## Cellule 6 — Schémas V4 + prompt généré
**V4** :
- `TCR` : 4 colonnes `debit_n`, `credit_n`, `debit_n1`, `credit_n1`
- `DECL` : nouveau schéma enrichi (~30 champs)
- `ACTIF` / `PASSIF` : inchangés

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — SCHÉMAS + PROMPT | BILANS_V4 | 2026-08-14
# V4 : TCR DEBIT/CREDIT + DECL enrichi
# ════════════════════════════════════════════════════════════
SCHEMAS = {
 'ACTIF': {'cols': ['brut','amort','n','n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'terrains': 'Terrains',
   'batiments': 'Batiments',
   'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession',
   'immobilisations_en_cours': 'Immobilisations en cours',
   'titres_mis_en_equivalence': 'Titres mis en equivalence',
   'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises',
   'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours',
   'clients': 'Clients',
   'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots & Assimiles',
   'autres_creances_assimiles': 'Autres Creances & Emplois assimiles',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants',
   'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT',
   'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'cols': ['n','n1'], 'postes': {
   'capital_emis': 'Capital emis (ou compte de l exploitant)',
   'capital_non_appele': 'Capital non appele',
   'primes_reserves': 'Primes et reserves (Reserves consolidees)',
   'ecart_reevaluation': 'Ecart de reevaluation',
   'ecart_equivalence': 'Ecart d equivalence (1)',
   'resultat_net_passif': 'Resultat net (Resultat net part du groupe) (1)',
   'report_a_nouveau': 'Autres capitaux propres - Report a nouveau',
   'part_societe_consolidante': 'Part de la societe consolidante (1)',
   'part_minoritaires': 'Part des minoritaires (1)',
   'total_capitaux_propres': 'TOTAL I',
   'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots differes et provisionnes',
   'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits comptabilises d avance',
   'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
   'impots_passif': 'Impots',
   'autres_dettes': 'Autres dettes',
   'tresorerie_passif': 'Tresorerie Passif',
   'total_passifs_courants': 'TOTAL PASSIFS COURANTS (II ou III)',
   'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'cols': ['debit_n','credit_n','debit_n1','credit_n1'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises',
   'produits_fabriques': 'Produits Fabriques',
   'prestations_services': 'Prestations de Services',
   'ventes_travaux': 'Ventes de Travaux',
   'produits_annexes': 'Produits Annexes',
   'rabais_remises_ristournes_accordes': 'Rabais, remises, ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net des Rabais, remises, ristournes',
   'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee',
   'subvention_exploitation': 'Subvention d exploitation',
   'production_exercice': 'I-Production de l exercice',
   'achats_marchandises_vendues': 'Achats de Marchandises vendues',
   'matieres_premieres': 'Matieres premieres',
   'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks',
   'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
   'autres_consommations': 'Autres consommations',
   'rabais_remises_obtenus_achats': 'Rabais, remises, ristournes obtenus sur achats',
   'sous_traitance_generale': 'Sous-traitance generale',
   'locations': 'Locations',
   'entretien_reparations': 'Entretien, reparations et maintenance',
   'primes_assurances': 'Primes d assurances',
   'personnel_exterieur': 'Personnel exterieur a l entreprise',
   'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
   'publicite': 'Publicite',
   'deplacements_missions': 'Deplacements, missions et receptions',
   'autres_services': 'Autres services',
   'rabais_remises_obtenus_services': 'Rabais, remises, ristournes obtenus sur services exterieurs',
   'consommations_exercice': 'II-Consommations de l exercice',
   'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation (I-II)',
   'charges_personnel': 'Charges de personnel',
   'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
   'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
   'autres_produits_operationnels': 'Autres produits operationnels',
   'autres_charges_operationnelles': 'Autres charges operationnelles',
   'dotations_amortissements': 'Dotations aux amortissements',
   'provisions': 'Provisions',
   'pertes_valeur': 'Perte de Valeur',
   'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
   'resultat_operationnel': 'V-Resultat operationnel',
   'produits_financiers': 'Produits financiers',
   'charges_financieres': 'Charges financieres',
   'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire (V+VI)',
   'elements_extraordinaires_produits': 'Elements extraordinaires (Produits)',
   'elements_extraordinaires_charges': 'Elements extraordinaires (Charges)',
   'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats',
   'impots_differes_resultats': 'Impots differes (variations) sur resultats',
   'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
 'DECL': {'cols': ['valeur'], 'postes': {
   'nif': 'Numero d Identification Fiscale',
   'article_imposition': 'Article d imposition',
   'nin': 'Numero d Identification National',
   'raison_sociale': 'Designation de l entreprise',
   'activite_principale': 'Activite principale',
   'code_activite': 'Code Activite',
   'registre_commerce': 'Registre de Commerce',
   'adresse_siege': 'Adresse siege social',
   'representant_legal': 'Representant legal',
   'cabinet_comptable': 'Cabinet de comptabilite',
   'cac_cabinet': 'Certification des comptes - Cabinet',
   'cac_nom': 'Certification des comptes - Nom CAC',
   'cac_agrement': 'Certification des comptes - Agrement',
   'exercice_annee': 'Resultat de l exercice - Annee',
   'exercice_periode_debut': 'Periode d imposition - Debut',
   'exercice_periode_fin': 'Periode d imposition - Fin',
   'annee_souscription': 'Annee de souscription',
   'chiffre_affaires_global_ht': 'Chiffre d affaires global hors taxes',
   'resultat_comptable': 'Resultat comptable',
   'resultat_comptable_type': 'Resultat comptable type (Benefice/Deficit)',
   'resultat_fiscal': 'Resultat fiscal',
   'resultat_fiscal_type': 'Resultat fiscal type (Benefice/Deficit)'}},
}

# ── Prompt généré depuis les schémas ─────────────────────────
L = []
L.append('Lis cette page d\'un dossier fiscal algerien (imprime Serie G).')
L.append('')
L.append('ETAPE 1 — Identifie le tableau principal :')
L.append('- ACTIF  : titre "BILAN (ACTIF)"')
L.append('- PASSIF : titre "BILAN (PASSIF)"')
L.append('- TCR    : titre "COMPTE DE RESULTAT"')
L.append('- DECL   : page "DECLARATION DE L IMPOT SUR LES BENEFICES DES SOCIETES" (entete "Annee de souscription")')
L.append('- AUTRE  : toute autre page (annexes, tableaux, TAP, page vide)')
L.append('')
L.append('ETAPE 2 — Extrais les montants du tableau identifie en JSON :')
L.append('{"type": ..., "entreprise": ..., "exercice": ..., "nif": ..., "annee_souscription": ..., "postes": {cle: {col: montant}}}')
L.append('- entreprise : raison sociale en haut de page | exercice : date de cloture (ex 31/12/2025) | nif : NIF si visible sinon null')
L.append('- annee_souscription : uniquement sur page DECL (ex 2026), sinon null')
L.append('- Colonnes : ACTIF → brut, amort, n, n1 | PASSIF → n, n1')
L.append('- TCR : 4 colonnes par poste : debit_n, credit_n, debit_n1, credit_n1')
L.append('  * debit_n  = montant DEBIT exercice N')
L.append('  * credit_n = montant CREDIT exercice N')
L.append('  * debit_n1 = montant DEBIT exercice N-1')
L.append('  * credit_n1 = montant CREDIT exercice N-1')
L.append('  * Un montant ne peut etre que dans UNE seule colonne (DEBIT ou CREDIT). Si vide : null.')
L.append('- DECL : extrais les champs listes ci-dessous (valeur ou null)')
L.append('- Montant entre parentheses = negatif : (1 553 799) → -1553799')
L.append('- Montants en NOMBRES JSON sans espaces ni separateurs ; case vide ou illisible : null')
L.append('- Utilise EXACTEMENT les cles ci-dessous. Si AUTRE : {"type": "AUTRE"}')
for table, spec in SCHEMAS.items():
    L.append(f'--- Si {table} ---')
    for key, label in spec['postes'].items():
        L.append(f'{key} : ligne "{label}"')
L.append('')
L.append('PRECISIONS SCAN :')
L.append('- Les pages peuvent etre inclinees : lis normalement malgre l inclinaison.')
L.append('- Des cachets, tampons, signatures peuvent recouvrir du texte : ignore-les et lis la valeur IMPRIMEE dessous.')
L.append('- entreprise, nif et exercice doivent etre lus en haut de CHAQUE page.')
L.append('')
L.append('REGLES : JSON valide uniquement, sans texte avant/apres, pas de markdown, pas de backticks, aucun champ invente.')
PROMPT_BILAN = '\n'.join(L)
print(f'✅ Schémas + prompt OK ({len(PROMPT_BILAN)} caractères)')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6bis — PROMPT CLASSIF + SORTIE ALLÉGÉE | BILANS_V4
# ════════════════════════════════════════════════════════════
PROMPT_CLASSIF = ('Page dun dossier fiscal algerien Serie G. Reponds UN seul mot : '
                  'ACTIF si titre BILAN (ACTIF) ; PASSIF si BILAN (PASSIF) ; '
                  'TCR si COMPTE DE RESULTAT ; DECL si page DECLARATION (Annee de souscription) ; '
                  'AUTRE sinon.')

PROMPT_BILAN += ('\n- IMPORTANT : dans postes, retourne UNIQUEMENT les cles avec une valeur '
                  'non nulle (les cles absentes seront mises a null automatiquement).')
print('✅ Two-pass prêt')

## Cellule 7 — Normalisation V4
**V4** : ajout `normalise_decl` + gestion TCR 4 colonnes.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — NORMALISATION | BILANS_V4 | 2026-08-14
# V4 : + normalise_decl ; TCR 4 colonnes
# ════════════════════════════════════════════════════════════
def norm_montant(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub(r'[^\d.,]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif ',' in s: s = s.replace(',', '')
    elif s.count('.') > 1: s = s.replace('.', '')
    try: return -float(s) if neg else float(s)
    except Exception: return None

def norm_str(v):
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null','none','n/a') else None

def norm_upper(v):
    s = norm_str(v)
    return s.upper() if s else None

def norm_nif(v):
    s = norm_str(v)
    return re.sub(r'[^0-9]', '', s) if s else None

def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall(r'20\d{2}', s)
    return m[-1] if m else None

def norm_date_fr(v):
    s = norm_str(v)
    if not s: return None
    m = re.search(r'(\d{1,2})/(\d{1,2})/(\d{4})', s)
    return f"{m.group(1).zfill(2)}/{m.group(2).zfill(2)}/{m.group(3)}" if m else s

def norm_type_resultat(v):
    s = norm_str(v)
    if not s: return None
    sl = s.lower()
    if 'benef' in sl or 'bénéf' in sl: return 'Benefice'
    if 'defic' in sl or 'défic' in sl or 'perte' in sl: return 'Deficit'
    return None

def normalise_table(table, data):
    spec = SCHEMAS[table]
    raw = data.get('postes') or {}
    out = {}
    for key in spec['postes']:
        vals = raw.get(key)
        vals = vals if isinstance(vals, dict) else {}
        for col in spec['cols']:
            out[f'{key}_{col}'] = norm_montant(vals.get(col))
    return out

def normalise_decl(data):
    raw = data.get('postes') or {}
    decl = {}
    decl['nif'] = norm_nif(raw.get('nif'))
    decl['article_imposition'] = norm_str(raw.get('article_imposition'))
    decl['nin'] = norm_nif(raw.get('nin'))
    decl['raison_sociale'] = norm_str(raw.get('raison_sociale'))
    decl['activite_principale'] = norm_str(raw.get('activite_principale'))
    decl['code_activite'] = norm_str(raw.get('code_activite'))
    decl['registre_commerce'] = norm_str(raw.get('registre_commerce'))
    decl['adresse_siege'] = norm_str(raw.get('adresse_siege'))
    decl['representant_legal'] = norm_str(raw.get('representant_legal'))
    decl['cabinet_comptable'] = norm_str(raw.get('cabinet_comptable'))
    decl['cac_cabinet'] = norm_str(raw.get('cac_cabinet'))
    decl['cac_nom'] = norm_str(raw.get('cac_nom'))
    decl['cac_agrement'] = norm_str(raw.get('cac_agrement'))
    decl['certifie_par_cac'] = bool(decl['cac_cabinet'] or decl['cac_nom'])
    decl['exercice_annee'] = norm_annee4(raw.get('exercice_annee'))
    decl['exercice_periode_debut'] = norm_date_fr(raw.get('exercice_periode_debut'))
    decl['exercice_periode_fin'] = norm_date_fr(raw.get('exercice_periode_fin'))
    decl['annee_souscription'] = norm_annee4(raw.get('annee_souscription'))
    decl['chiffre_affaires_global_ht'] = norm_montant(raw.get('chiffre_affaires_global_ht'))
    decl['resultat_comptable'] = norm_montant(raw.get('resultat_comptable'))
    decl['resultat_comptable_type'] = norm_type_resultat(raw.get('resultat_comptable_type'))
    decl['resultat_fiscal'] = norm_montant(raw.get('resultat_fiscal'))
    decl['resultat_fiscal_type'] = norm_type_resultat(raw.get('resultat_fiscal_type'))
    return decl

print('✅ Normalisation V4 OK')

## Cellule 8 — Cohérence inter-pages (anti-fraude / anti-mélange)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — CONTRÔLE COHÉRENCE | BILANS_V4 | 2026-08-14
# Même client / même NIF / même année sur toutes les pages
# ════════════════════════════════════════════════════════════
def norm_identite(s):
    s = norm_upper(s)
    return re.sub(r'[^A-Z0-9]', '', s) if s else None

def meme_entreprise(a, b):
    if not a or not b: return True
    if a == b or a in b or b in a: return True
    return difflib.SequenceMatcher(None, a, b).ratio() > 0.80

def controle_coherence(parsed, annee_attendue=None):
    anomalies, excluded = [], set()
    idents = [(p['index'], norm_identite(p['meta'].get('entreprise'))) for p in parsed]
    vals = [e for _, e in idents if e]
    ref_ent = max(set(vals), key=lambda e: sum(1 for x in vals if meme_entreprise(e, x))) if vals else None
    for idx, e in idents:
        if ref_ent and e and not meme_entreprise(ref_ent, e):
            anomalies.append(f'PAGE {idx+1}: CLIENT DIFFERENT')
            excluded.add(idx)
    annees = [(p['index'], norm_annee4(p['meta'].get('exercice'))) for p in parsed]
    yy = [a for _, a in annees if a]
    ref_annee = max(set(yy), key=yy.count) if yy else None
    for idx, a in annees:
        if ref_annee and a and a != ref_annee:
            anomalies.append(f'PAGE {idx+1}: ANNEE {a} ≠ {ref_annee}')
    if annee_attendue and ref_annee and ref_annee != str(annee_attendue):
        anomalies.append(f'ANNEE DOSSIER {ref_annee} ≠ ATTENDUE {annee_attendue}')
    nifs = [(p['index'], norm_nif(p['meta'].get('nif'))) for p in parsed]
    nn = [n for _, n in nifs if n and len(n) >= 12]
    ref_nif = max(set(nn), key=nn.count) if nn else None
    for idx, n in nifs:
        if ref_nif and n and len(n) >= 12 and n != ref_nif:
            anomalies.append(f'PAGE {idx+1}: NIF DIFFERENT')
    return ref_ent, ref_nif, ref_annee, anomalies, excluded

print('✅ Contrôle cohérence OK')

## Cellule 9 — Export Excel V4 (2 lignes par bilan)
**V4** : affichage TCR en DEBIT/CREDIT + colonne DECL.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — EXPORT EXCEL | BILANS_V4 | 2026-08-14
# 2 lignes/bilan (N, N-1) ; TCR debit/credit ; bloc DECL
# ════════════════════════════════════════════════════════════
COLORS = {'META':'FFD6E4F0','ACTIF':'FFE2EFDA','PASSIF':'FFDAE3F3','TCR':'FFFCE4D6','DECL':'FFE4DFEC'}
HDRS   = {'META':'FF1F4E79','ACTIF':'FF375623','PASSIF':'FF203864','TCR':'FF833C00','DECL':'FF4B2D73'}
TOT_ACT = ['total_actif_non_courant','total_actif_courant','total_general_actif']

def build_cols():
    cols = [('META', k, lab) for k, lab in [
        ('fichier','Fichier'),('entreprise','Entreprise'),('nif','NIF'),
        ('exercice','Exercice'),('annee_exercice','Année exercice'),
        ('annee_depot','Année dépôt'),('date_traitement','Date traitement'),
        ('temps_total_s','Temps (s)'),('tokens_in','Tokens IN'),
        ('tokens_out','Tokens OUT'),('tokens_total','Tokens total'),
        ('pages_trouvees','Tableaux trouvés'),('anomalies','Anomalies')]]
    for key in SCHEMAS['ACTIF']['postes']: cols.append(('ACTIF', key, key))
    for key in TOT_ACT:
        cols.append(('ACTIF', key+'_brut', key+' [brut]'))
        cols.append(('ACTIF', key+'_amort', key+' [amort]'))
    for key in SCHEMAS['PASSIF']['postes']: cols.append(('PASSIF', key, key))
    for key in SCHEMAS['TCR']['postes']:
        cols.append(('TCR', key+'_debit', key+' [D]'))
        cols.append(('TCR', key+'_credit', key+' [C]'))
    for key in SCHEMAS['DECL']['postes']: cols.append(('DECL', key, key))
    return cols

def val_for(d, groupe, key, suf):
    if groupe == 'DECL':
        return (d.get('DECL') or {}).get(key)
    t = d.get(groupe) or {}
    if groupe == 'ACTIF' and (key.endswith('_brut') or key.endswith('_amort')):
        return t.get(key) if suf == 'n' else None
    if groupe == 'TCR':
        base = key.rsplit('_', 1)[0]
        sens = key.rsplit('_', 1)[1]
        return t.get(f'{base}_{sens}_{suf}')
    return t.get(f'{key}_{suf}')

def create_excel(path, rows):
    all_cols = build_cols()
    wb = Workbook(); ws = wb.active; ws.title = 'Bilans'
    grp = defaultdict(list); idx = 1
    for g, _, _ in all_cols: grp[g].append(idx); idx += 1
    for g, cs in grp.items():
        s, e = cs[0], cs[-1]
        if s < e: ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s); c.value = g
        c.font = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill = PatternFill('solid', start_color=HDRS[g])
        c.alignment = Alignment(horizontal='center', vertical='center')
    for i, (g, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i); c.value = label
        c.font = Font(bold=True, name='Arial', size=8)
        c.fill = PatternFill('solid', start_color=COLORS[g])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        ws.column_dimensions[get_column_letter(i)].width = 16
    ws.row_dimensions[2].height = 30
    ws.freeze_panes = ws.cell(row=3, column=7)
    rn = 3
    for d in rows:
        annee_n = norm_str(d.get('exercice'))
        annee_n1 = str(int(annee_n)-1) if annee_n and annee_n.isdigit() else None
        for exer, annee, suf in [('N', annee_n, 'n'), ('N-1', annee_n1, 'n1')]:
            for ci, (g, key, _) in enumerate(all_cols, start=1):
                if g == 'META':
                    if key == 'exercice': val = exer
                    elif key == 'annee_exercice': val = annee
                    else: val = d.get(key)
                else:
                    val = val_for(d, g, key, suf)
                c = ws.cell(row=rn, column=ci); c.value = val
                c.font = Font(name='Arial', size=9)
                c.fill = PatternFill('solid', start_color=COLORS[g])
                if isinstance(val, float): c.number_format = '#,##0.00'
                if g == 'META' and key == 'anomalies' and val:
                    c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')
            rn += 1
    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} bilans → {rn-3} lignes | {len(all_cols)} colonnes')

print('✅ Export Excel V4 OK')

## Cellule 10 — Log

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — LOG | BILANS_V4 | 2026-08-14
# ════════════════════════════════════════════════════════════
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(ligne + '\n')
print('✅ Log OK')

## Cellule 11 — Pipeline V4
**V4** : extraction DECL enrichie + TCR 4 colonnes + contrôles arithmétiques.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 11 — PIPELINE V4 | BILANS_V4 | 2026-08-14
# V4 : DECL enrichie + TCR DEBIT/CREDIT + contrôles arithmétiques
# ════════════════════════════════════════════════════════════
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log(f'À traiter : {len(a_traiter)} | déjà traités : {len(deja)}')

t_total = time.time(); n_ok = n_err = 0
TYPES = {'ACTIF', 'PASSIF', 'TCR'}

def parse_type(text):
    t = (text or '').upper()
    for k in ('ACTIF', 'PASSIF', 'TCR', 'DECL'):
        if k in t: return k
    return 'AUTRE'

def controles_arithmetiques(actif, passif, tcr):
    anomalies = []
    # ACTIF : brut - amort ≈ net (par poste, exercice N)
    for key in SCHEMAS['ACTIF']['postes']:
        b, a, n = actif.get(f'{key}_brut'), actif.get(f'{key}_amort'), actif.get(f'{key}_n')
        if b is not None and n is not None:
            amort = a or 0.0
            if abs(b - amort - n) > 1:
                anomalies.append(f'ACTIF {key}: brut-amort≠net ({b:,.0f}-{amort:,.0f}≠{n:,.0f})')
    # ACTIF : somme sections = total général
    for suf, lab in [('n','N'),('n1','N-1')]:
        tnc = actif.get(f'total_actif_non_courant_{suf}')
        tc  = actif.get(f'total_actif_courant_{suf}')
        tg  = actif.get(f'total_general_actif_{suf}')
        if tnc is not None and tc is not None and tg is not None:
            if abs(tnc + tc - tg) > 1:
                anomalies.append(f'ACTIF [{lab}]: sections≠total')
    # PASSIF : I + II + III = total
    for suf, lab in [('n','N'),('n1','N-1')]:
        i  = passif.get(f'total_capitaux_propres_{suf}')
        ii = passif.get(f'total_passifs_non_courants_{suf}')
        iii= passif.get(f'total_passifs_courants_{suf}')
        tg = passif.get(f'total_general_passif_{suf}')
        if None not in (i, ii, iii, tg) and abs(i + ii + iii - tg) > 1:
            anomalies.append(f'PASSIF [{lab}]: I+II+III≠total')
    # TCR : RN crédit - RN débit = RN passif
    rn_credit = tcr.get('resultat_net_exercice_credit_n')
    rn_debit  = tcr.get('resultat_net_exercice_debit_n')
    rn_passif = passif.get('resultat_net_passif_n')
    rn_tcr = (rn_credit or 0) - (rn_debit or 0)
    if rn_passif is not None and (rn_credit is not None or rn_debit is not None):
        if abs(rn_tcr - rn_passif) > 1:
            anomalies.append(f'TCR: RN {rn_tcr:,.0f} ≠ Passif {rn_passif:,.0f}')
    return anomalies

for num, pdf_path in enumerate(a_traiter, start=1):
    t0 = time.time()
    try:
        pages   = pdf_to_pages(pdf_path)
        actives = [p for p in pages if not is_blank(p['image'])]
        tok_in = tok_out = 0

        # PASS 1 : classification
        mini = [resize(p['image'], 600) for p in actives]
        reps1 = []
        for bs in range(0, len(mini), 16):
            reps1 += ask_batch(PROMPT_CLASSIF, mini[bs:bs+16])
        tok_in  += sum(r['tokens_in']  for r in reps1)
        tok_out += sum(r['tokens_out'] for r in reps1)
        utiles = [(p, parse_type(r['text'])) for p, r in zip(actives, reps1)
                  if parse_type(r['text']) != 'AUTRE']

        # PASS 2 : extraction
        parsed, decl_data, annee_depot = [], None, None
        for bs in range(0, len(utiles), GPU_BATCH_SIZE):
            batch = utiles[bs:bs+GPU_BATCH_SIZE]
            reps  = ask_batch(PROMPT_BILAN, [p['image'] for p, _ in batch])
            for (page, tpage), rep in zip(batch, reps):
                tok_in += rep['tokens_in']; tok_out += rep['tokens_out']
                data = parse_json(rep['text'])
                t = data.get('type', tpage)
                if t not in TYPES and t != 'DECL': continue
                meta = {k: data.get(k) for k in ('entreprise', 'nif')}
                meta['exercice'] = None if t == 'DECL' else data.get('exercice')
                if t == 'DECL':
                    decl_data = data
                    dep = norm_annee4(data.get('annee_souscription'))
                    if dep and not annee_depot: annee_depot = dep
                parsed.append({'index': page['index'], 'type': t, 'meta': meta, 'data': data})
            gc.collect(); torch.cuda.empty_cache()

        # Cohérence + fusion
        ref_ent, ref_nif, ref_annee, anomalies, excluded = controle_coherence(parsed, ANNEE_ATTENDUE)
        ent_raw = next((norm_str(p['meta'].get('entreprise')) for p in parsed
                        if p['index'] not in excluded
                        and meme_entreprise(ref_ent or '', norm_identite(p['meta'].get('entreprise')))), None)
        tables, tcr_pages, doublons = {}, [], []
        for p in parsed:
            if p['index'] in excluded or p['type'] == 'DECL': continue
            t = p['type']
            if t == 'TCR': tcr_pages.append(p['data'])
            else:
                if t in tables: doublons.append(t); continue
                tables[t] = normalise_table(t, p['data'])
        tcr = {}
        for d in tcr_pages:
            for k, v in normalise_table('TCR', d).items():
                if tcr.get(k) is None and v is not None: tcr[k] = v
        if tcr: tables['TCR'] = tcr
        if doublons: anomalies.append('DOUBLON: ' + ', '.join(doublons))

        # DECL enrichie
        decl_norm = normalise_decl(decl_data) if decl_data else {}

        # Contrôles arithmétiques V4
        anomalies += controles_arithmetiques(
            tables.get('ACTIF', {}), tables.get('PASSIF', {}), tables.get('TCR', {}))

        dt = round(time.time() - t0, 2)
        manquants = sorted(TYPES - set(tables.keys()))
        result = {
            'fichier': pdf_path.name,
            'entreprise': ent_raw, 'nif': ref_nif,
            'exercice': ref_annee, 'annee_depot': annee_depot,
            'date_traitement': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'temps_total_s': dt, 'tokens_in': tok_in, 'tokens_out': tok_out,
            'tokens_total': tok_in + tok_out,
            'pages_trouvees': ', '.join(sorted(tables.keys())),
            'anomalies': ' | '.join(anomalies) if anomalies else None,
            'DECL': decl_norm,
            **{t: tables.get(t, {}) for t in TYPES},
        }
        with open(JSON_DIR / f'{pdf_path.stem}.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        n_ok += 1
        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        msg = f'[{num: >4}/{len(a_traiter)}] ✅ {pdf_path.name} | {dt:.1f}s | tok={tok_in+tok_out} | {result["pages_trouvees"]}'
        if manquants:         msg += f' | ⚠️ manquants: {", ".join(manquants)}'
        if result['anomalies']: msg += f' | 🔴 {result["anomalies"]}'
        log(msg + f' | ETA {eta/3600:.1f}h')
    except Exception as e:
        n_err += 1
        log(f'[{num: >4}/{len(a_traiter)}] ❌ {pdf_path.name} — {e}')
        continue

log('Génération Excel...')
rows = [json.load(open(jf, encoding='utf-8')) for jf in sorted(JSON_DIR.glob('*.json'))]
create_excel(EXCEL_PATH, rows)
log(f'✅ Terminé en {time.time()-t_total:.1f}s | OK {n_ok} | Erreurs {n_err}')

## Cellule 12 — Contrôles comptables V4

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 12 — CONTRÔLES COMPTABLES | BILANS_V4 | 2026-08-14
# Actif=Passif ; RN TCR=RN passif ; DECL↔TCR↔Passif
# ════════════════════════════════════════════════════════════
for d in rows:
    a, p, t, decl = d.get('ACTIF') or {}, d.get('PASSIF') or {}, d.get('TCR') or {}, d.get('DECL') or {}
    # Actif = Passif
    for suf, lab in [('n', 'N'), ('n1', 'N-1')]:
        ta, tp = a.get(f'total_general_actif_{suf}'), p.get(f'total_general_passif_{suf}')
        if ta and tp and abs(ta - tp) > 1:
            print(f'❌ {d["fichier"]} [{lab}] : Actif {ta:,.0f} ≠ Passif {tp:,.0f}')
    # RN TCR = RN passif
    rn_credit = t.get('resultat_net_exercice_credit_n')
    rn_debit  = t.get('resultat_net_exercice_debit_n')
    rn_tcr = (rn_credit or 0) - (rn_debit or 0)
    rn_p = p.get('resultat_net_passif_n')
    if rn_p is not None and (rn_credit is not None or rn_debit is not None) and abs(rn_tcr - rn_p) > 1:
        print(f'❌ {d["fichier"]} : RN TCR {rn_tcr:,.0f} ≠ Passif {rn_p:,.0f}')
    # CA DECL ≈ CA TCR
    ca_decl = decl.get('chiffre_affaires_global_ht')
    ca_tcr  = t.get('chiffre_affaires_net_credit_n')
    if ca_decl and ca_tcr and abs(ca_decl - ca_tcr) > 1:
        print(f'⚠️ {d["fichier"]} : CA DECL {ca_decl:,.0f} ≠ CA TCR {ca_tcr:,.0f}')
    # Résultat comptable DECL ≈ RN passif
    rc_decl = decl.get('resultat_comptable')
    if rc_decl and rn_p and abs(rc_decl - rn_p) > 1:
        print(f'⚠️ {d["fichier"]} : Résultat comptable DECL {rc_decl:,.0f} ≠ RN passif {rn_p:,.0f}')
print('🔎 Contrôles V4 terminés (aucune ligne = cohérent)')

## Cellule 13 — Data Dictionary

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 13 — DATA DICTIONARY | BILANS_V4 | 2026-08-14
# ════════════════════════════════════════════════════════════
dd_path = OUTPUT_DIR / 'data_dictionary_bilans_v4.csv'
with open(dd_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['table', 'cle', 'libelle_imprime', 'colonnes'])
    for t, spec in SCHEMAS.items():
        for k, lab in spec['postes'].items():
            w.writerow([t, k, lab, '|'.join(spec['cols'])])
print(f'✅ Data dictionary : {dd_path} ({sum(len(s["postes"]) for s in SCHEMAS.values())} postes)')

## Cellule 14 — Générateur FORFAIT `.xlsx` (NOUVEAU V4)
**Objectif** : produire `FORFAIT Cas 1_{client}_{année}.xlsx` par bilan.
- Copie du template avec `shutil.copy2` (préserve formules, styles, feuilles)
- Remplissage `Saisie actif`, `Saisie passif`, `Saisie TCR` via `openpyxl`
- Conversion dinars → **KDZD** (÷1000, arrondi 2 décimales)
- N-2 toujours vide ; brut/amort N-1 vides (non disponibles dans l'imprimé)
- Les formules des feuilles `Etats fin reclassés` et `Décision` restent intactes (recalcul à l'ouverture)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 14 — GÉNÉRATEUR FORFAIT | BILANS_V4 | 2026-08-14
# Copie template .xlsx + remplissage Saisie actif/passif/TCR
# Formules 'Etats fin reclassés' et 'Décision' préservées
# ════════════════════════════════════════════════════════════

# ─── Mapping libellés FORFAIT ↔ clés pipeline ───────────────
FORFAIT_ACTIF_MAPPING = {
    'ecarts_acquisition_goodwill': ['Ecart d acquisition', 'goodwill'],
    'immobilisations_incorporelles': ['Immobilisations incorporelles'],
    'terrains': ['Terrains'],
    'batiments': ['Bâtiments', 'Batiments'],
    'autres_immobilisations_corporelles': ['Autres immobilisations corporelles'],
    'immobilisations_en_concession': ['Immobilisations en concession'],
    'immobilisations_en_cours': ['Immobilisations en cours'],
    'titres_mis_en_equivalence': ['Titres mis en équivalence'],
    'autres_participations_creances': ['Autres participations'],
    'autres_titres_immobilises': ['Autres titres immobilisés'],
    'prets_actifs_financiers_non_courants': ['Prêts et autres actifs financiers'],
    'impots_differes_actif': ['Impôts différés actif'],
    'total_actif_non_courant': ['TOTAL ACTIF NON COURANT'],
    'stocks_encours': ['Stocks et encours'],
    'clients': ['Clients'],
    'autres_debiteurs': ['Autres débiteurs'],
    'impots_assimiles_actif': ['Impôts et assimilés'],
    'autres_creances_assimiles': ['Autres créances et emplois assimilés'],
    'placements_financiers_courants': ['Placements et autres actifs financiers courants'],
    'tresorerie_actif': ['Trésorerie'],
    'total_actif_courant': ['TOTAL ACTIF COURANT'],
    'total_general_actif': ['TOTAL GENERAL ACTIF', 'TOTAL GÉNÉRAL ACTIF'],
}

FORFAIT_PASSIF_MAPPING = {
    'capital_emis': ['Capital émis'],
    'capital_non_appele': ['Capital non appelé'],
    'primes_reserves': ['Primes et réserves'],
    'ecart_reevaluation': ['Ecarts de réévaluation'],
    'ecart_equivalence': ['Ecart d équivalence'],
    'resultat_net_passif': ['Résultat net'],
    'report_a_nouveau': ['Report à nouveau'],
    'part_societe_consolidante': ['Part de la société consolidante'],
    'part_minoritaires': ['Part des minoritaires'],
    'total_capitaux_propres': ['TOTAL I'],
    'emprunts_dettes_financieres': ['Emprunts et dettes financières'],
    'impots_differes_provisionnes': ['Impôts (différés'],
    'autres_dettes_non_courantes': ['Autres dettes non courantes'],
    'provisions_produits_avance': ['Provisions et produits'],
    'total_passifs_non_courants': ['TOTAL II'],
    'fournisseurs_rattaches': ['Fournisseurs'],
    'impots_passif': ['Impôts'],
    'autres_dettes': ['Autres dettes'],
    'tresorerie_passif': ['Trésorerie Passif'],
    'total_passifs_courants': ['TOTAL III'],
    'total_general_passif': ['TOTAL PASSIF'],
}

FORFAIT_TCR_MAPPING = {
    'ventes_marchandises': ['Ventes de marchandises'],
    'produits_fabriques': ['Produits fabriqués'],
    'prestations_services': ['Prestations de services'],
    'ventes_travaux': ['Vente de travaux'],
    'produits_annexes': ['Produits annexes'],
    'rabais_remises_ristournes_accordes': ['Rabais, remises, ristournes accordés'],
    'chiffre_affaires_net': ['Chiffre d affaires net'],
    'production_stockee_destockee': ['Production stockée'],
    'production_immobilisee': ['Production immobilisée'],
    'subvention_exploitation': ['Subventions d exploitation'],
    'production_exercice': ['I-Production de l exercice'],
    'achats_marchandises_vendues': ['Achats de marchandises vendues'],
    'matieres_premieres': ['Matières premières'],
    'autres_approvisionnements': ['Autres approvisionnements'],
    'variation_stocks': ['Variations des stocks'],
    'achats_etudes_prestations': ['Achats d études'],
    'autres_consommations': ['Autres consommations'],
    'rabais_remises_obtenus_achats': ['Rabais, remises, ristournes obtenus sur achats'],
    'sous_traitance_generale': ['Sous-traitance générale'],
    'locations': ['Locations'],
    'entretien_reparations': ['Entretien, réparations'],
    'primes_assurances': ['Primes d assurances'],
    'personnel_exterieur': ['Personnel extérieur'],
    'remuneration_intermediaires': ['Rémunération d intermédiaires'],
    'publicite': ['Publicité'],
    'deplacements_missions': ['Déplacements, missions'],
    'autres_services': ['Autres services'],
    'rabais_remises_obtenus_services': ['Rabais, remises, ristournes obtenus sur services'],
    'consommations_exercice': ['II-Consommations de l exercice'],
    'valeur_ajoutee_exploitation': ['III-Valeur ajoutée'],
    'charges_personnel': ['Charges de personnel'],
    'impots_taxes_assimiles': ['Impôts et taxes'],
    'excedent_brut_exploitation': ['IV-Excédent brut'],
    'autres_produits_operationnels': ['Autres produits opérationnels'],
    'autres_charges_operationnelles': ['Autres charges opérationnelles'],
    'dotations_amortissements': ['Dotations aux amortissements'],
    'provisions': ['Provision'],
    'pertes_valeur': ['Pertes de valeur'],
    'reprises_pertes_valeur_provisions': ['Reprise sur pertes'],
    'resultat_operationnel': ['V-Résultat opérationnel'],
    'produits_financiers': ['Produits financiers'],
    'charges_financieres': ['Charges financières'],
    'resultat_financier': ['VI-Résultat financier'],
    'resultat_ordinaire': ['VII-Résultat ordinaire'],
    'elements_extraordinaires_produits': ['Eléments extraordinaires (produits)'],
    'elements_extraordinaires_charges': ['Eléments extraordinaires (Charges)'],
    'resultat_extraordinaire': ['VIII-Résultat extraordinaire'],
    'impots_exigibles_resultats': ['Impôts exigibles'],
    'impots_differes_resultats': ['Impôts différés'],
    'resultat_net_exercice': ['RESULTAT NET DE L EXERCICE'],
}

def to_kdzd(v):
    if v is None: return None
    return round(v / 1000.0, 2)

def _clean_name(s):
    return re.sub(r'[^A-Za-z0-9 _-]', '', s or 'ENTREPRISE').strip()

def _find_row_by_label(ws, label_col, patterns, max_row=200):
    for r in range(1, min(ws.max_row, max_row) + 1):
        cell_val = ws.cell(row=r, column=label_col).value
        if cell_val is None: continue
        cell_str = str(cell_val).lower()
        for pat in patterns:
            if pat.lower() in cell_str:
                return r
    return None

def _safe_write(ws, row, col, value):
    if row is None or value is None: return
    ws.cell(row=row, column=col, value=value)

def generate_forfait(result):
    if not TEMPLATE_FORFAIT.exists():
        print(f'⚠️ Template introuvable : {TEMPLATE_FORFAIT}'); return None

    entreprise = _clean_name(result.get('entreprise'))
    annee = result.get('exercice') or 'XXXX'
    out_path = FORFAIT_OUTPUT_DIR / f'FORFAIT Cas 1_{entreprise}_{annee}.xlsx'

    try:
        shutil.copy2(TEMPLATE_FORFAIT, out_path)
        wb = load_workbook(out_path)
    except Exception as e:
        print(f'❌ Ouverture/création FORFAIT impossible : {e}'); return None

    actif  = result.get('ACTIF')  or {}
    passif = result.get('PASSIF') or {}
    tcr    = result.get('TCR')    or {}
    decl   = result.get('DECL')   or {}

    # ── Saisie actif ────────────────────────────────────────
    cfg_a = FORFAIT_COORDS['actif']
    if cfg_a['sheet'] in wb.sheetnames:
        ws = wb[cfg_a['sheet']]
        # En-tête
        r, c = cfg_a['nature_bilan']
        ws.cell(row=r, column=c, value='Fiscal')
        exerc = norm_str(result.get('exercice'))
        if exerc:
            r, c = cfg_a['date_arrete_n']
            ws.cell(row=r, column=c, value=exerc)
            m = re.match(r'(\d{2})/(\d{2})/(\d{4})', exerc)
            if m:
                n1 = f'{m.group(1)}/{m.group(2)}/{int(m.group(3))-1}'
                r, c = cfg_a['date_arrete_n1']
                ws.cell(row=r, column=c, value=n1)
        cac = decl.get('cac_cabinet') or decl.get('cac_nom')
        if cac:
            r, c = cfg_a['certifie_cac']
            ws.cell(row=r, column=c, value=cac)
        # Postes
        for key, patterns in FORFAIT_ACTIF_MAPPING.items():
            row = _find_row_by_label(ws, cfg_a['label_col'], patterns)
            if row is None: continue
            cn, cn1 = cfg_a['cols_n'], cfg_a['cols_n1']
            # N : brut / amort / net
            _safe_write(ws, row, cn['brut'],  to_kdzd(actif.get(f'{key}_brut')))
            _safe_write(ws, row, cn['amort'], to_kdzd(actif.get(f'{key}_amort')))
            _safe_write(ws, row, cn['net'],   to_kdzd(actif.get(f'{key}_n')))
            # N-1 : net uniquement (brut/amort non disponibles)
            _safe_write(ws, row, cn1['net'],  to_kdzd(actif.get(f'{key}_n1')))

    # ── Saisie passif ───────────────────────────────────────
    cfg_p = FORFAIT_COORDS['passif']
    if cfg_p['sheet'] in wb.sheetnames:
        ws = wb[cfg_p['sheet']]
        for key, patterns in FORFAIT_PASSIF_MAPPING.items():
            row = _find_row_by_label(ws, cfg_p['label_col'], patterns)
            if row is None: continue
            _safe_write(ws, row, cfg_p['col_n'],  to_kdzd(passif.get(f'{key}_n')))
            _safe_write(ws, row, cfg_p['col_n1'], to_kdzd(passif.get(f'{key}_n1')))

    # ── Saisie TCR (ventilation DEBIT/CREDIT) ───────────────
    cfg_t = FORFAIT_COORDS['tcr']
    if cfg_t['sheet'] in wb.sheetnames:
        ws = wb[cfg_t['sheet']]
        for key, patterns in FORFAIT_TCR_MAPPING.items():
            row = _find_row_by_label(ws, cfg_t['label_col'], patterns)
            if row is None: continue
            ctn, ctn1 = cfg_t['cols_n'], cfg_t['cols_n1']
            _safe_write(ws, row, ctn['debit'],  to_kdzd(tcr.get(f'{key}_debit_n')))
            _safe_write(ws, row, ctn['credit'], to_kdzd(tcr.get(f'{key}_credit_n')))
            _safe_write(ws, row, ctn1['debit'],  to_kdzd(tcr.get(f'{key}_debit_n1')))
            _safe_write(ws, row, ctn1['credit'], to_kdzd(tcr.get(f'{key}_credit_n1')))

    # Sauvegarder (formules des autres feuilles préservées)
    try:
        wb.save(out_path)
        wb.close()
        print(f'✅ FORFAIT : {out_path.name}')
        return out_path
    except Exception as e:
        print(f'❌ Sauvegarde FORFAIT impossible : {e}')
        return None

print('✅ Générateur FORFAIT défini')

## Cellule 15 — Exécution génération FORFAIT
À exécuter **après** la Cellule 11 (pipeline) — ou directement si les JSON existent déjà.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 15 — GÉNÉRATION FORFAIT | BILANS_V4 | 2026-08-14
# Boucle sur tous les JSON → FORFAIT Cas 1_{client}_{annee}.xlsx
# ════════════════════════════════════════════════════════════
json_files = sorted(JSON_DIR.glob('*.json'))
if not json_files:
    print('⚠️ Aucun JSON trouvé — exécute d\'abord la Cellule 11')
else:
    print(f'📁 {len(json_files)} JSON → génération FORFAIT .xlsx')
    n_gen = n_skip = 0
    for jf in json_files:
        try:
            data = json.load(open(jf, encoding='utf-8'))
            if not (data.get('ACTIF') or data.get('PASSIF') or data.get('TCR')):
                print(f'⏭️  {jf.stem} : aucune donnée financière → ignoré')
                n_skip += 1
                continue
            if generate_forfait(data):
                n_gen += 1
        except Exception as e:
            print(f'❌ {jf.stem} : {e}')
    print(f'\n✅ {n_gen} FORFAIT .xlsx générés | {n_skip} ignorés → {FORFAIT_OUTPUT_DIR}')

## Cellule 16 — Vérification coordonnées FORFAIT
**À exécuter UNE FOIS** après conversion du template pour valider/ajuster `FORFAIT_COORDS`.
Affiche le contenu des feuilles de saisie avec les numéros de lignes/colonnes.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 16 — VÉRIFICATION COORDONNÉES FORFAIT | BILANS_V4
# À exécuter UNE FOIS pour valider FORFAIT_COORDS
# ════════════════════════════════════════════════════════════
if not TEMPLATE_FORFAIT.exists():
    print(f'⚠️ Template introuvable : {TEMPLATE_FORFAIT}')
else:
    wb = load_workbook(TEMPLATE_FORFAIT, data_only=False)
    for sheet_name in ['Saisie actif', 'Saisie passif', 'Saisie TCR']:
        if sheet_name not in wb.sheetnames:
            print(f'⚠️ Feuille absente : {sheet_name}')
            continue
        ws = wb[sheet_name]
        print(f'\n{"="*70}')
        print(f'=== {sheet_name} ({ws.max_row} lignes × {ws.max_column} cols) ===')
        print(f'{"="*70}')
        for r in range(1, min(ws.max_row, 45) + 1):
            row_vals = []
            for c in range(1, min(ws.max_column, 12) + 1):
                v = ws.cell(row=r, column=c).value
                if v is not None and str(v).strip() != '':
                    row_vals.append((c, str(v)[:40]))
            if row_vals:
                print(f'  Ligne {r:3d}: {row_vals}')
    wb.close()
    print('\n✅ Vérification terminée — ajustez FORFAIT_COORDS si nécessaire')